# 04 — SimMIM + VICReg pretraining (v2, anti-collapse)

Two-task self-supervised pretraining of the MONAI 2D `SwinTransformer` on
fluorescence-microscopy patches:

- **Task A — SimMIM reconstruction.** Block-masked L1 reconstruction with
  foreground weighting (`l1_fg`), as in `02_pretrain_simmim_swin-*.ipynb`.
- **Task B — VICReg invariance/variance/covariance regularization** on the
  pooled encoder features of two augmented views.

Why two tasks: the v2 runs in `outputs/` show **representation collapse**
under pure-reconstruction objectives — `eff_rank` saturates at ≤ 8.8 / 768
(≤ 1.2 % of capacity) and `std_med` falls to ≈ 0.04 by epoch 100 in
simmim-A and simmim-D. VICReg's variance hinge `Σᵢ ReLU(1 − std(zᵢ))` is
a provably collapse-preventing pooled-level regulariser and is cheap to
add (Bardes et al., 2022, *VICReg: Variance-Invariance-Covariance
Regularization for Self-Supervised Learning*, ICLR).

## Design notes (deviations from the canonical recipes)

| | Canonical SimMIM / VICReg | This notebook |
|---|---|---|
| Domain | RGB ImageNet | 3-ch fluorescence MIPs, [0,1], sparse foreground |
| Backbone | Swin-B / RN50 | MONAI 2D Swin (embed=48, depths=[2,2,2,2]) |
| Recon loss | L1 over masked pixels | `simmim_l1_fg` — weight = 1 + α·1[x>τ], α≈99 |
| Augmentation | RandResizedCrop + ColorJitter + Blur | D4 + per-channel gain U(0.85,1.15) + N(0,σ²) — color/crop are unsafe on this domain |
| Mask per view | (VICReg has none) | **Same mask** across both views — keeps per-image foreground content shared |
| Projector | 8192-d (VICReg paper) | 1024-d (B=32 makes 8192 wasteful) |
| Loss weighting | 25 / 25 / 1 (sim/std/cov) | same; outer `w_recon=w_vicreg=1.0` |
| Logging | epoch-level CSV | epoch CSV + per-step `steps.jsonl` + recon PNGs + resume from `last.pt` |


## 1. Imports & runtime hygiene

In [ ]:
from __future__ import annotations

import csv
import gc
import json
import math
import os
import random
import sys
import time
import itertools
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# --- runtime hygiene (auto-patched) ---
import warnings as _warnings
_warnings.filterwarnings(
    "ignore", category=FutureWarning,
    message=r".*cuda\.cudart.*",
)
_prev_unraisablehook = sys.unraisablehook
def _silence_dataloader_teardown(unraisable):
    e = unraisable.exc_value
    if isinstance(e, AssertionError) and "can only test a child process" in str(e):
        return
    if callable(_prev_unraisablehook):
        _prev_unraisablehook(unraisable)
sys.unraisablehook = _silence_dataloader_teardown
# --- end runtime hygiene ---

REPO_ROOT = Path.cwd()
while REPO_ROOT.name and not (REPO_ROOT / "data_utils").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print(f"repo root: {REPO_ROOT}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")


## 2. Configuration

`train_cfg.w_recon` and `train_cfg.w_vicreg` are **outer** weights; VICReg's
internal `lambda_sim/std/cov = 25/25/1` come from the paper. With both outer
weights at 1.0 the per-step loss magnitudes are typically dominated by the
variance hinge during the first ~10 epochs (which is exactly the goal — the
model is forced out of any collapsed init).

In [ ]:
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

cfg = SimpleNamespace(
    seed         = 42,
    output_root  = REPO_ROOT / "outputs",
    tag          = "simmim_vicreg_v2",
    run_id       = RUN_TS,
)

data_cfg = SimpleNamespace(
    patches_dir = REPO_ROOT / "data" / "patches_128",
    split_json  = REPO_ROOT / "data" / "splits" / "v2_split.json",
    batch_size  = 32,
    num_workers = 2,
    pin_memory  = True,
)

model_cfg = SimpleNamespace(
    in_channels       = 3,
    spatial_dims      = 2,
    img_size          = 128,
    feature_size      = 48,
    patch_size        = 2,
    window_size       = 7,
    dropout_path_rate = 0.0,
    use_checkpoint    = False,
    # VICReg projector: D_enc=feature_size*2**4=768 → projector_dim
    projector_dim     = 1024,
    projector_hidden  = 1024,
)

train_cfg = SimpleNamespace(
    # SimMIM
    mask_ratio        = 0.4,
    mask_block_size   = 8,
    masking_strategy  = "grid",        # "grid" | "content_aware" | "cutout"
    fg_mask_ratio     = 0.0,
    loss_kind         = "l1_fg",       # "l1" | "l1_fg" | "l1_l2_mix"
    fg_alpha_override = None,
    # VICReg
    lambda_sim        = 25.0,
    lambda_std        = 25.0,
    lambda_cov        = 1.0,
    w_recon           = 1.0,
    w_vicreg          = 1.0,
    # View augmentation (symmetric, mild)
    view_gain_lo      = 0.85,
    view_gain_hi      = 1.15,
    view_noise_std    = 0.02,
    # Optim
    lr                = 3e-5,
    weight_decay      = 0.05,
    warmup_epochs     = 10,
    epochs            = 200,
    grad_clip_norm    = 5.0,
    # Logging / checkpoints
    diag_every        = 1,             # epochs (was 5 in earlier notebooks)
    recon_png_every   = 10,
    n_diag_batches    = 8,
    silhouette_k      = 10,
    silhouette_n_max  = 1024,
)

torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
print(json.dumps({
    "tag": cfg.tag, "run_id": cfg.run_id,
    "model": vars(model_cfg), "train": vars(train_cfg),
}, default=str, indent=2))


## 3. Dataset

Reuses the v2 split produced by `00_data_and_split.ipynb`.
`fg_stats.tau` and `.alpha` come straight from that split JSON.

In [ ]:
from data_utils.split_patch_dataset import SplitPatchDataset

assert data_cfg.split_json.exists(), \
    f"{data_cfg.split_json} not found. Run 00_data_and_split.ipynb first."

ds_train = SplitPatchDataset(data_cfg.patches_dir, data_cfg.split_json, which="train")
ds_val   = SplitPatchDataset(data_cfg.patches_dir, data_cfg.split_json, which="val")

train_loader = DataLoader(
    ds_train, batch_size=data_cfg.batch_size, shuffle=True,
    num_workers=data_cfg.num_workers,
    persistent_workers=(data_cfg.num_workers > 0),
    pin_memory=data_cfg.pin_memory, drop_last=True,
)
val_loader = DataLoader(
    ds_val, batch_size=data_cfg.batch_size, shuffle=False,
    num_workers=data_cfg.num_workers,
    persistent_workers=(data_cfg.num_workers > 0),
    pin_memory=data_cfg.pin_memory,
)
print(f"train: {len(ds_train):,} | val: {len(ds_val):,}")

TAU   = float(ds_train.fg_stats["tau"])
ALPHA = float(train_cfg.fg_alpha_override) if train_cfg.fg_alpha_override is not None \
        else float(ds_train.fg_stats["alpha"])
print(f"TAU={TAU:.5f} ALPHA={ALPHA:.2f}")


## 4. Model — `SimMIMVICRegSwin`

Reuses the SimMIM head from `02_pretrain_simmim_swin-A`, plus a 3-layer MLP
projector for the VICReg branch.

- `encode(x)`        → spatial features  (B, 768, 4, 4)
- `encode_pooled(x)` → GAP-pooled features (B, 768) — used for VICReg & diagnostics
- `project(z_pool)`  → projector output (B, projector_dim) — VICReg loss is on this
- `forward(x, mask)` → (recon, z_pool) tuple

In [ ]:
from monai.networks.nets.swin_unetr import SwinTransformer


class SimMIMVICRegSwin(nn.Module):
    def __init__(self, args):
        super().__init__()
        self.in_chans       = args.in_channels
        self.img_size       = args.img_size
        self.encoder_stride = args.patch_size * (2 ** 4)
        assert args.img_size % self.encoder_stride == 0

        patch_size  = (args.patch_size,)  * args.spatial_dims
        window_size = (args.window_size,) * args.spatial_dims

        self.swinViT = SwinTransformer(
            in_chans       = args.in_channels,
            embed_dim      = args.feature_size,
            window_size    = window_size,
            patch_size     = patch_size,
            depths         = [2, 2, 2, 2],
            num_heads      = [3, 6, 12, 24],
            mlp_ratio      = 4.0,
            qkv_bias       = True,
            drop_rate      = 0.0,
            attn_drop_rate = 0.0,
            drop_path_rate = args.dropout_path_rate,
            norm_layer     = nn.LayerNorm,
            use_checkpoint = args.use_checkpoint,
            spatial_dims   = args.spatial_dims,
        )

        enc_ch = args.feature_size * (2 ** 4)
        self.enc_dim = enc_ch

        # SimMIM "linear" head (1x1 Conv + PixelShuffle).
        self.decoder = nn.Sequential(
            nn.Conv2d(enc_ch, self.encoder_stride ** 2 * self.in_chans, kernel_size=1),
            nn.PixelShuffle(self.encoder_stride),
        )

        # DEVIATION: pixel-space mask token (per-channel scalar broadcast).
        self.mask_token = nn.Parameter(torch.zeros(1, self.in_chans, 1, 1))
        nn.init.trunc_normal_(self.mask_token, mean=0.0, std=0.02)

        # VICReg projector: 3-layer MLP with BN + ReLU between linears.
        h = args.projector_hidden
        d = args.projector_dim
        self.projector = nn.Sequential(
            nn.Linear(enc_ch, h, bias=False),
            nn.BatchNorm1d(h),
            nn.ReLU(inplace=True),
            nn.Linear(h, h, bias=False),
            nn.BatchNorm1d(h),
            nn.ReLU(inplace=True),
            nn.Linear(h, d, bias=True),
        )

    def encode(self, x):
        return self.swinViT(x.contiguous())[4]

    def encode_pooled(self, x):
        return self.encode(x).mean(dim=(2, 3))

    def project(self, z_pool):
        return self.projector(z_pool)

    def forward(self, x, mask):
        x_masked = x * (1.0 - mask) + self.mask_token * mask
        z_spatial = self.encode(x_masked)             # (B, 768, 4, 4)
        recon     = self.decoder(z_spatial)           # (B, C, H, W)
        z_pool    = z_spatial.mean(dim=(2, 3))        # (B, 768)
        return recon, z_pool


model = SimMIMVICRegSwin(model_cfg).to(device)
n_param = sum(p.numel() for p in model.parameters())
print(f"params: {n_param/1e6:.2f}M | enc_dim={model.enc_dim} | "
      f"projector_dim={model_cfg.projector_dim}")


## 5. Reconstruction loss (same as `02_pretrain_simmim_swin`)

In [ ]:
def simmim_l1_loss(pred, target, mask):
    err   = (pred - target).abs() * mask
    denom = mask.sum() * pred.shape[1] + 1e-8
    return err.sum() / denom


def simmim_l1_fg_loss(pred, target, mask, tau: float, alpha: float):
    fg_mask = (target > tau).float()
    weight  = 1.0 + alpha * fg_mask
    err     = (pred - target).abs() * mask * weight
    denom   = (mask * weight).sum() + 1e-8
    return err.sum() / denom


def simmim_l1_l2_mix_loss(pred, target, mask):
    diff  = (pred - target) * mask
    l1    = diff.abs().sum()
    l2    = (diff * diff).sum()
    denom = mask.sum() * pred.shape[1] + 1e-8
    return 0.5 * (l1 / denom) + 0.5 * (l2 / denom)


def reconstruction_loss(pred, target, mask, kind: str, tau: float, alpha: float):
    if kind == "l1":         return simmim_l1_loss(pred, target, mask)
    if kind == "l1_fg":      return simmim_l1_fg_loss(pred, target, mask, tau, alpha)
    if kind == "l1_l2_mix":  return simmim_l1_l2_mix_loss(pred, target, mask)
    raise ValueError(f"unknown loss_kind: {kind!r}")


## 6. VICReg loss (fp32)

`L_vicreg = λ_sim · MSE(z¹,z²) + λ_std · ½(hinge(z¹)+hinge(z²)) + λ_cov · ½(cov(z¹)+cov(z²))`

with `hinge(z) = mean_i ReLU(1 − std(z_i) − ε)` and
`cov(z) = (1/d) Σ_{i≠j} cov_matrix(z)_{ij}²`. Computation is forced to fp32
because covariance/variance terms are numerically more fragile than the L1
recon under AMP autocast — see existing `outputs/` runs which show 11–23 %
of epochs hitting non-finite grads.

In [ ]:
def _off_diag_sq_sum(c: torch.Tensor) -> torch.Tensor:
    n = c.shape[0]
    return c.pow(2).sum() - c.diagonal().pow(2).sum()


def vicreg_loss(z1: torch.Tensor, z2: torch.Tensor,
                lam_sim: float, lam_std: float, lam_cov: float,
                eps: float = 1e-4) -> dict:
    """Bardes/Ponce/LeCun 2022. Inputs are projector outputs (B, D)."""
    # Force fp32 even if upstream forward ran under autocast.
    z1 = z1.float()
    z2 = z2.float()
    B, D = z1.shape

    # Invariance.
    l_sim = F.mse_loss(z1, z2)

    # Variance hinge — the explicit anti-collapse term.
    std1 = torch.sqrt(z1.var(dim=0) + eps)
    std2 = torch.sqrt(z2.var(dim=0) + eps)
    l_std = (F.relu(1.0 - std1).mean() + F.relu(1.0 - std2).mean()) * 0.5

    # Covariance redundancy reduction.
    z1c = z1 - z1.mean(dim=0)
    z2c = z2 - z2.mean(dim=0)
    cov1 = (z1c.T @ z1c) / max(B - 1, 1)
    cov2 = (z2c.T @ z2c) / max(B - 1, 1)
    l_cov = (_off_diag_sq_sum(cov1) + _off_diag_sq_sum(cov2)) / (2.0 * D)

    total = lam_sim * l_sim + lam_std * l_std + lam_cov * l_cov
    return {"total": total, "sim": l_sim, "std": l_std, "cov": l_cov}


## 7. View augmentation + mask sampling

**Symmetric** mild augmentation for both views (rubber-duck advice — asymmetric
view stats bias the invariance signal). Each view independently gets a D4
element + per-channel multiplicative gain + Gaussian noise. The *mask is shared*
between views (sparse-foreground patches mean independent 40 % masks would
remove different rare structures and degrade the invariance signal).

In [ ]:
def _d4_one(x: torch.Tensor) -> torch.Tensor:
    """Apply a random D4 element to a single CHW tensor."""
    if torch.rand(()) > 0.5: x = x.flip(-1)
    if torch.rand(()) > 0.5: x = x.flip(-2)
    k = int(torch.randint(0, 4, ()).item())
    if k > 0: x = torch.rot90(x, k, dims=(-2, -1))
    return x


def make_view(x: torch.Tensor, gain_lo: float, gain_hi: float, noise_std: float
              ) -> torch.Tensor:
    """Symmetric, mild fluorescence-safe augmentation. Operates batch-wise
    on (B, C, H, W). Returns a tensor with same shape, clipped to [0,1]."""
    B, C, H, W = x.shape
    out = torch.empty_like(x)
    for i in range(B):
        out[i] = _d4_one(x[i])
    # Per-(B,C) multiplicative gain.
    gain = torch.empty(B, C, 1, 1, device=x.device).uniform_(gain_lo, gain_hi)
    out = out * gain
    if noise_std > 0:
        out = out + torch.randn_like(out) * noise_std
    return out.clamp_(0.0, 1.0)


def random_block_mask(img, block_size: int, mask_ratio: float):
    B, _, H, W = img.shape
    assert H % block_size == 0 and W % block_size == 0
    gh, gw = H // block_size, W // block_size
    n_blocks = gh * gw
    n_mask   = int(math.ceil(n_blocks * mask_ratio))
    noise    = torch.rand(B, n_blocks, device=img.device)
    rank     = noise.argsort(dim=1)
    flat     = (rank < n_mask).float()
    m        = flat.view(B, 1, gh, gw)
    return F.interpolate(m, scale_factor=block_size, mode="nearest")


def content_aware_block_mask(img, block_size, mask_ratio, tau,
                             fg_mask_ratio=0.5, fg_block_thresh=0.05):
    B, C, H, W = img.shape
    assert H % block_size == 0 and W % block_size == 0
    gh, gw = H // block_size, W // block_size
    n_blocks = gh * gw
    n_mask   = int(math.ceil(n_blocks * mask_ratio))
    n_fg_q   = int(math.floor(n_mask * fg_mask_ratio))

    fg_pix      = (img > tau).any(dim=1, keepdim=True).float()
    fg_block    = F.avg_pool2d(fg_pix, kernel_size=block_size)
    fg_block    = (fg_block >= fg_block_thresh).float().view(B, n_blocks)

    flat = torch.zeros(B, n_blocks, device=img.device)
    for b in range(B):
        fg_idx = torch.nonzero(fg_block[b] > 0.5, as_tuple=False).flatten()
        bg_idx = torch.nonzero(fg_block[b] < 0.5, as_tuple=False).flatten()
        n_fg_b = min(n_fg_q, fg_idx.numel())
        # Sample fg blocks first.
        if n_fg_b > 0:
            sel = fg_idx[torch.randperm(fg_idx.numel(), device=img.device)[:n_fg_b]]
            flat[b, sel] = 1.0
        # Fill the rest from any non-selected block (uniform).
        n_remaining = n_mask - n_fg_b
        if n_remaining > 0:
            pool = torch.cat([
                fg_idx[n_fg_b:] if fg_idx.numel() > n_fg_b else torch.empty(0, dtype=torch.long, device=img.device),
                bg_idx,
            ])
            if pool.numel() > 0:
                sel = pool[torch.randperm(pool.numel(), device=img.device)[:n_remaining]]
                flat[b, sel] = 1.0

    m = flat.view(B, 1, gh, gw)
    return F.interpolate(m, scale_factor=block_size, mode="nearest")


def cutout_mask(img, block_size, mask_ratio):
    """One contiguous square hole per image, area ≈ mask_ratio * H*W."""
    B, _, H, W = img.shape
    side = int(round(math.sqrt(mask_ratio * H * W)))
    side = max(block_size, (side // block_size) * block_size)
    m = torch.zeros(B, 1, H, W, device=img.device)
    for b in range(B):
        y = torch.randint(0, H - side + 1, ()).item()
        x = torch.randint(0, W - side + 1, ()).item()
        m[b, 0, y:y+side, x:x+side] = 1.0
    return m


def sample_mask(img: torch.Tensor) -> torch.Tensor:
    s = train_cfg.masking_strategy
    if s == "grid":
        return random_block_mask(img, train_cfg.mask_block_size, train_cfg.mask_ratio)
    if s == "content_aware":
        return content_aware_block_mask(
            img, train_cfg.mask_block_size, train_cfg.mask_ratio,
            tau=TAU, fg_mask_ratio=train_cfg.fg_mask_ratio,
        )
    if s == "cutout":
        return cutout_mask(img, train_cfg.mask_block_size, train_cfg.mask_ratio)
    raise ValueError(f"unknown masking_strategy: {s!r}")


In [ ]:
def viz_recon(model, x, mask, indices, suptitle, *, header=None,
              autocast=True):
    """Run inference, print per-patch in/out-of-mask MAE, then plot a
    4-row grid: input | mask | recon | |err| with a cyan mask contour
    on the |err| row and a per-axis colorbar (vmax shared across the
    batch so colours mean the same thing across patches).

    Pure -- does not modify the model state beyond the eval/train flag.
    Caller is responsible for any state save/restore (see overfit cell).
    """
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad(), torch.amp.autocast(
            device.type, enabled=autocast and device.type == "cuda"
        ):
            out = model(x, mask)
            recon = (out[0] if isinstance(out, tuple) else out).float()
    finally:
        if was_training:
            model.train()

    err = (recon - x).abs()
    if header:
        print(header)
    for j, idx in enumerate(indices):
        in_m = mask[j, 0]
        e    = err[j].mean(0)
        in_mae  = (e * in_m).sum() / max(in_m.sum().item(), 1.0)
        out_mae = (e * (1.0 - in_m)).sum() / max((1.0 - in_m).sum().item(), 1.0)
        print(f"  idx={idx}: in-mask MAE={in_mae.item():.5f}  "
              f"out-of-mask MAE={out_mae.item():.5f}  "
              f"full MAE={err[j].mean().item():.5f}")

    err_max = max(float(err.max().item()), 1e-6)
    n = x.shape[0]
    fig, axes = plt.subplots(4, n, figsize=(3.6 * n, 12.0))
    if n == 1:
        axes = axes[:, None]
    for j in range(n):
        o  = x[j].clamp(0, 1).cpu()
        rr = recon[j].clamp(0, 1).cpu()
        mm = mask[j, 0].cpu()
        ee = err[j].mean(0).cpu()
        o_disp = o.permute(1, 2, 0).numpy()  if o.shape[0]  >= 3 else o[0].numpy()
        r_disp = rr.permute(1, 2, 0).numpy() if rr.shape[0] >= 3 else rr[0].numpy()

        axes[0, j].imshow(o_disp, cmap="gray")
        axes[0, j].set_title(f"input idx={indices[j]}")
        axes[0, j].axis("off")

        axes[1, j].imshow(mm.numpy(), cmap="gray", vmin=0, vmax=1)
        axes[1, j].set_title(f"mask cov={mask[j].mean().item():.2f}")
        axes[1, j].axis("off")

        axes[2, j].imshow(r_disp, cmap="gray")
        axes[2, j].set_title("recon")
        axes[2, j].axis("off")

        im = axes[3, j].imshow(ee.numpy(), cmap="hot",
                               vmin=0.0, vmax=err_max)
        axes[3, j].contour(mm.numpy(), levels=[0.5],
                           colors=["cyan"], linewidths=1.0)
        in_mae  = (ee * mm).sum() / max(mm.sum().item(), 1.0)
        out_mae = (ee * (1.0 - mm)).sum() / max((1.0 - mm).sum().item(), 1.0)
        axes[3, j].set_title(
            f"|err|  in={in_mae.item():.3f}  out={out_mae.item():.3f}"
        )
        axes[3, j].axis("off")
        cb = plt.colorbar(im, ax=axes[3, j], fraction=0.046, pad=0.04)
        cb.set_label("|err|", fontsize=8)
        cb.ax.tick_params(labelsize=7)
    plt.suptitle(suptitle)
    plt.tight_layout()
    plt.show()

## 8. Sanity check — two views + shared mask

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(cfg.seed)
_x = next(iter(train_loader)).to(device)
_v1 = make_view(_x, train_cfg.view_gain_lo, train_cfg.view_gain_hi, train_cfg.view_noise_std)
_v2 = make_view(_x, train_cfg.view_gain_lo, train_cfg.view_gain_hi, train_cfg.view_noise_std)
_m  = sample_mask(_x)
print(f"x:{tuple(_x.shape)}  mask coverage: {_m.mean().item()*100:.1f}%")

fig, axes = plt.subplots(3, 4, figsize=(10, 7))
for j in range(4):
    axes[0, j].imshow(_x[j].permute(1, 2, 0).clamp(0, 1).cpu()); axes[0, j].set_title("orig")
    axes[1, j].imshow(_v1[j].permute(1, 2, 0).clamp(0, 1).cpu()); axes[1, j].set_title("view 1")
    axes[2, j].imshow(_v2[j].permute(1, 2, 0).clamp(0, 1).cpu()); axes[2, j].set_title("view 2")
    for ax in axes[:, j]: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


## V3 Sanity checks (pre-training)

Uses the **same fixed patch indices** so reconstructions are directly comparable
across notebooks. Verifies dataset shape/dtype/normalization, the actual
augmented inputs the model will see, and (for masked methods) that the mask
matches the configured ratio.


### Definitions for visualisations

In [ ]:
# === V3-SANITY-CHECKS PRE ===
# Pre-training sanity checks. Indices match the reference notebook
# notebooks/training/pretrain_simMIM_swin_v2_fixed.ipynb so plots are
# directly comparable across runs.
import matplotlib.pyplot as plt
import numpy as np
import torch

from types import SimpleNamespace
overfit_cfg    = SimpleNamespace(patch_indices=[3, 4, 50, 1000])
unseen_idx     = 100
unseen_indices = [200, 500, 1500, 2000]

def _resolve_fixed(ds, indices):
    """Return a (len(indices), C, H, W) tensor of the requested patches.

    SplitPatchDataset stores the underlying PatchDataset as `_base`, so we
    bypass the split filter to get the same absolute patches the reference
    notebook uses (matches `dataset[i]` in pretrain_simMIM_swin_v2_fixed.ipynb).
    """
    base = getattr(ds, "_base", ds)
    return torch.stack([base[i] for i in indices]).to(device)


x_fixed = _resolve_fixed(ds_train, overfit_cfg.patch_indices)
print(f"x_fixed: shape={tuple(x_fixed.shape)}  dtype={x_fixed.dtype}  "
      f"device={x_fixed.device}")

# Per-channel intensity stats on a real train batch -- catches channel
# misalignment (e.g. a channel that is identically zero) and abnormal
# normalisation. Reference: cells 4 and 21 of pretrain_simMIM_swin_v2_fixed.ipynb.
_x = next(iter(train_loader)).to(device)
print(f"\ntrain batch:  shape={tuple(_x.shape)}  dtype={_x.dtype}  "
      f"min={_x.min():.3f}  max={_x.max():.3f}")
for c in range(_x.shape[1]):
    xc = _x[:, c]
    print(f"  ch{c}:  min={xc.min():.4f}  max={xc.max():.4f}  "
          f"mean={xc.mean():.4f}  std={xc.std():.4f}")

assert _x.ndim == 4, f"expected (B,C,H,W), got {_x.shape}"
assert _x.shape[1] == model_cfg.in_channels, (
    f"channel mismatch: dataset={_x.shape[1]} model={model_cfg.in_channels}"
)
assert _x.shape[2] == model_cfg.img_size and _x.shape[3] == model_cfg.img_size, (
    f"spatial mismatch: dataset={_x.shape[2:]} model={model_cfg.img_size}"
)
assert -1e-6 <= float(_x.min()) and float(_x.max()) <= 1.0 + 1e-6, "values outside [0,1]"

# Foreground sparsity diagnostic. With tau ~ 0.66 and alpha = 99 the loss
# is dominated by the tiny fraction of foreground pixels; this prints the
# realised fraction so the bug is visible at glance.
try:
    _fg = (_x.amax(dim=1, keepdim=True) >= TAU).float()
    print(f"  foreground fraction (>= TAU={float(TAU):.4f}): "
          f"{_fg.mean().item()*100:.2f}%")
except (NameError, RuntimeError):
    pass

# --- Visualise the 4 fixed patches: per-channel slices + RGB composite
n_idx = x_fixed.shape[0]; C = x_fixed.shape[1]
fig, axes = plt.subplots(C + 1, n_idx, figsize=(3.0 * n_idx, 3.0 * (C + 1)))
if axes.ndim == 1:
    axes = axes[None, :]
for j in range(n_idx):
    img = x_fixed[j].cpu()
    for c in range(C):
        axes[c, j].imshow(img[c].numpy(), cmap="gray", vmin=0, vmax=1)
        axes[c, j].set_title(f"idx={overfit_cfg.patch_indices[j]}  ch{c}")
        axes[c, j].axis("off")
    if C >= 3:
        rgb = img[:3].clamp(0, 1).permute(1, 2, 0).numpy()
    else:
        rgb = np.repeat(img.numpy(), 3, axis=0).transpose(1, 2, 0).clip(0, 1)
    axes[C, j].imshow(rgb)
    axes[C, j].set_title("RGB" if C >= 3 else "expanded gray")
    axes[C, j].axis("off")
plt.suptitle(f"Pre-train fixed indices: {overfit_cfg.patch_indices}")
plt.tight_layout(); plt.show()

# --- VICReg uses two augmented *views* of the same patch + a shared mask
torch.manual_seed(cfg.seed)
v1 = make_view(x_fixed, train_cfg.view_gain_lo, train_cfg.view_gain_hi, train_cfg.view_noise_std)
v2 = make_view(x_fixed, train_cfg.view_gain_lo, train_cfg.view_gain_hi, train_cfg.view_noise_std)
mask_fixed = sample_mask(x_fixed)
print(f"effective mask coverage: {mask_fixed.mean().item():.4f}  "
      f"(configured mask_ratio={train_cfg.mask_ratio})")
assert abs(mask_fixed.mean().item() - train_cfg.mask_ratio) < 0.20, (
    f"mask coverage drifted: realised={mask_fixed.mean().item():.3f} "
    f"target={train_cfg.mask_ratio:.3f}")

n_idx = x_fixed.shape[0]
fig, axes = plt.subplots(4, n_idx, figsize=(3.0 * n_idx, 12.0))
for j in range(n_idx):
    o  = x_fixed[j].clamp(0, 1).cpu()
    a1 = v1[j].clamp(0, 1).cpu()
    a2 = v2[j].clamp(0, 1).cpu()
    m  = mask_fixed[j, 0].cpu()
    axes[0, j].imshow(o.permute(1, 2, 0).numpy() if o.shape[0] >= 3 else o[0].numpy(), cmap="gray")
    axes[0, j].set_title(f"orig idx={overfit_cfg.patch_indices[j]}"); axes[0, j].axis("off")
    axes[1, j].imshow(a1.permute(1, 2, 0).numpy() if a1.shape[0] >= 3 else a1[0].numpy(), cmap="gray")
    axes[1, j].set_title("view 1"); axes[1, j].axis("off")
    axes[2, j].imshow(a2.permute(1, 2, 0).numpy() if a2.shape[0] >= 3 else a2[0].numpy(), cmap="gray")
    axes[2, j].set_title("view 2"); axes[2, j].axis("off")
    axes[3, j].imshow(m.numpy(), cmap="gray", vmin=0, vmax=1)
    axes[3, j].set_title(f"shared mask cov={mask_fixed[j].mean().item():.2f}"); axes[3, j].axis("off")
plt.tight_layout(); plt.show()

In [ ]:

# --- End-to-end forward shape check (no backprop)
model.eval()
with torch.no_grad():
    out = model(v1, mask_fixed)
    _r = out[0] if isinstance(out, tuple) else out
    _z = out[1] if isinstance(out, tuple) and len(out) >= 2 else None
print(f"recon shape: {tuple(_r.shape)}  dtype={_r.dtype}")
if _z is not None:
    print(f"z_pool shape: {tuple(_z.shape)}  dtype={_z.dtype}")
assert _r.shape == x_fixed.shape, f"recon {_r.shape} != input {x_fixed.shape}"


In [ ]:
# === V4-RECON-VIZ-HELPERS ===
# Single visualisation function used after the overfit-on-batch sanity
# check (cell below) AND after full training (post-training section), on
# the SAME 4 overfit patches + 3 unseen random train patches both times.
# The two figures are directly comparable -- if the final-training figure
# is no better than the overfit one, the full run did not learn anything
# that the 4-patch overfit didn't already memorise.
#
# This cell also picks the 3 unseen indices + their masks once, so all
# downstream cells share the same inputs (deterministic via cfg.seed).
import copy

import matplotlib.pyplot as plt
import numpy as np
import torch


def _resolve_patches(ds, indices, device):
    """Bypass the split filter on SplitPatchDataset and return a
    (len(indices), C, H, W) tensor of the requested absolute patches."""
    base = getattr(ds, "_base", ds)
    return torch.stack([base[int(i)] for i in indices]).to(device)


# Deterministic pick of 3 unseen random patches (excludes the overfit
# indices). Reused by both the overfit-recon-viz cell and the post-
# training viz cell so the two figures show the same patches.
_excluded = set(int(i) for i in overfit_cfg.patch_indices)
_base_train = getattr(ds_train, "_base", ds_train)
_rng = np.random.default_rng(cfg.seed + 17)
_pool = [i for i in range(len(_base_train)) if i not in _excluded]
viz_other_indices = [int(x) for x in _rng.choice(_pool, size=3, replace=False)]
x_other = _resolve_patches(ds_train, viz_other_indices, device)

# Deterministic mask for x_other so the SAME pixels are masked in the
# overfit-viz and the final-viz -- crucial for fair comparison.
torch.manual_seed(cfg.seed + 23)
mask_other = sample_mask(x_other)
print(f"viz: fixed indices = {overfit_cfg.patch_indices}  "
      f"unseen indices = {viz_other_indices}  "
      f"(out of {len(_base_train)} train patches)")

### Actuall overfit

IMPORTANT: RUN_OVERFIT_CHECK   = True   # set to False to skip


In [ ]:
# === V4-OVERFIT-ON-BATCH ===
# Overfit-on-a-batch sanity check (matches reference cells 25-29 of
# pretrain_simMIM_swin_v2_fixed.ipynb).
#
# This step trains the *current* model on the 4 fixed patches for a short
# burst with a saved/restored state_dict so the main training run starts
# from the same weights it would have without this cell.
import copy

RUN_OVERFIT_CHECK   = True   # set to False to skip
N_OVERFIT_STEPS     = 200
OVERFIT_LR          = 1e-4

In [ ]:

if RUN_OVERFIT_CHECK:
    _saved = copy.deepcopy(model.state_dict())
    opt = torch.optim.AdamW(model.parameters(), lr=OVERFIT_LR, weight_decay=0.0)
    losses = []
    model.train()
    for step in range(N_OVERFIT_STEPS):
        out = model(x_fixed, mask_fixed)
        r   = out[0] if isinstance(out, tuple) else out
        diff = (r - x_fixed).abs()
        denom = mask_fixed.sum() * x_fixed.shape[1] + 1e-8
        loss = (diff * mask_fixed).sum() / denom
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        losses.append(float(loss.item()))

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(losses)
    ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.set_yscale("log")
    ax.set_title(
        f"overfit-on-{x_fixed.shape[0]}-patches  "
        f"start={losses[0]:.4f}  end={losses[-1]:.4f}"
    )
    plt.tight_layout(); plt.show()

    drop = (losses[0] - losses[-1]) / max(losses[0], 1e-8)
    print(f"loss drop over {N_OVERFIT_STEPS} steps: {drop*100:.1f}%  "
          f"(>=50% expected for a working pipeline)")
    if drop < 0.50:
        print("  WARNING: model failed to overfit a tiny batch; investigate "
              "loss / mask / fp16 overflow before launching full training.")
else:
    print("overfit-on-batch sanity check skipped (RUN_OVERFIT_CHECK=False)")


In [ ]:
# === V4-OVERFIT-RECON-VIZ ===
# Visualise what the overfit-on-batch model learned: reconstructions on
# the same 4 patches it was trained on (expect near-perfect on masked
# pixels) plus 3 unseen random train-split patches (expect poor recon --
# a 4-patch overfit is OOD on the rest of the dataset). The same
# `viz_recon()` helper is reused in the post-training section, on the
# SAME indices, so the two figures are directly comparable.
if RUN_OVERFIT_CHECK:
    viz_recon(
        model, x_fixed, mask_fixed, overfit_cfg.patch_indices,
        suptitle=f"OVERFIT recon -- {x_fixed.shape[0]} TRAIN patches "
                 f"(expect near-perfect on masked pixels)",
        header="\nOVERFIT patches (expect near-perfect on masked pixels):",
    )

    viz_recon(
        model, x_other, mask_other, viz_other_indices,
        suptitle="OVERFIT recon -- 3 UNSEEN random train patches "
                 "(expect poor recon: 4-patch overfit is OOD)",
        header="\nUNSEEN random patches (expect poor recon -- 4-patch overfit, OOD):",
    )
else:
    print("overfit-recon visualisation skipped (RUN_OVERFIT_CHECK=False)")

In [ ]:
if RUN_OVERFIT_CHECK:
  model.load_state_dict(_saved)
  print("model state restored to its pre-sanity weights")

## 9. Optimizer + scheduler

AdamW (β=(0.9, 0.999), wd=0.05) per SimMIM Sec 4.1.1; linear warmup + cosine
decay. The scheduler is stepped per-epoch.

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_cfg.lr, weight_decay=train_cfg.weight_decay, betas=(0.9, 0.999),
)


def lr_lambda(epoch):
    if epoch < train_cfg.warmup_epochs:
        return (epoch + 1) / max(1, train_cfg.warmup_epochs)
    progress = (epoch - train_cfg.warmup_epochs) / max(1, train_cfg.epochs - train_cfg.warmup_epochs)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.amp.GradScaler(enabled=device.type == "cuda")


## 10. Diagnostics

- **eff_rank, n_dead, std_med** on pooled encoder features (same as previous
  notebooks; pooled features, not projector outputs).
- **mae_all / mae_fg / mae_bg** + per-channel MAE.
- **Clustering proxies** on pooled features: K-means **silhouette** and
  **Davies-Bouldin** at `k=silhouette_k` over up to `silhouette_n_max`
  val patches. Used as a *secondary* signal — silhouette rewards easy
  geometric partitioning, not necessarily biological structure.

In [ ]:
try:
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score, davies_bouldin_score
    _SKLEARN_OK = True
except ModuleNotFoundError:
    _SKLEARN_OK = False
    print("WARN: scikit-learn not installed; silhouette/davies_bouldin will be NaN. "
          "Install with: pip install scikit-learn")


@torch.no_grad()
def collect_pooled_features(model, loader, n_batches: int):
    model.eval()
    feats = []
    for i, x in enumerate(loader):
        if i >= n_batches: break
        x = x.to(device, non_blocking=True)
        feats.append(model.encode_pooled(x).cpu())
    if not feats:
        return torch.zeros(0, model.enc_dim)
    return torch.cat(feats, dim=0)


def collapse_metrics(feats: torch.Tensor) -> dict:
    if feats.shape[0] < 2:
        return dict(eff_rank=float("nan"), r_max=0, n_dead=-1, D=feats.shape[1] if feats.ndim==2 else -1, std_med=float("nan"))
    D = feats.shape[1]
    r_max = min(feats.shape[0] - 1, D)
    fs = feats.std(dim=0)
    n_dead = int((fs < 1e-3).sum().item())
    fc = feats - feats.mean(dim=0, keepdim=True)
    s = torch.linalg.svdvals(fc.float())
    s2 = (s ** 2) / ((s ** 2).sum() + 1e-12)
    eff_rank = torch.exp(-(s2 * torch.log(s2 + 1e-12)).sum()).item()
    return dict(eff_rank=eff_rank, r_max=r_max, n_dead=n_dead, D=D, std_med=float(fs.median()))


def cluster_metrics(feats: torch.Tensor, k: int, n_max: int, seed: int) -> dict:
    n = min(feats.shape[0], n_max)
    if not _SKLEARN_OK or n < max(k * 2, 20):
        return dict(silhouette=float("nan"), davies_bouldin=float("nan"), n_used=n, k=k)
    X = feats[:n].numpy()
    km = KMeans(n_clusters=k, n_init=4, random_state=seed)
    labels = km.fit_predict(X)
    if len(set(labels)) < 2:
        return dict(silhouette=float("nan"), davies_bouldin=float("nan"), n_used=n, k=k)
    return dict(
        silhouette=float(silhouette_score(X, labels)),
        davies_bouldin=float(davies_bouldin_score(X, labels)),
        n_used=int(n), k=int(k),
    )


@torch.no_grad()
def recon_mae_split(model, loader, tau, n_batches: int = 4):
    model.eval()
    s_all=0.0; n_all=0; s_fg=0.0; n_fg=0; s_bg=0.0; n_bg=0
    s_per_ch = None; n_per_ch = None
    for x in itertools.islice(loader, n_batches):
        x = x.to(device, non_blocking=True)
        m = sample_mask(x)
        with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
            r, _ = model(x, m)
        r = r.float()
        err = (r - x).abs()
        s_all += err.sum().item(); n_all += err.numel()
        fg = (x > tau)
        s_fg += err[fg].sum().item(); n_fg += int(fg.sum().item())
        bg_mask = ~fg
        s_bg += err[bg_mask].sum().item(); n_bg += int(bg_mask.sum().item())
        # Per-channel MAE
        per_ch = err.mean(dim=(0, 2, 3)).cpu()
        if s_per_ch is None:
            s_per_ch = per_ch.clone(); n_per_ch = 1
        else:
            s_per_ch += per_ch; n_per_ch += 1
    out = dict(
        mae_all = s_all / max(1, n_all),
        mae_fg  = s_fg  / max(1, n_fg),
        mae_bg  = s_bg  / max(1, n_bg),
    )
    if s_per_ch is not None:
        per = (s_per_ch / max(1, n_per_ch)).tolist()
        for i, v in enumerate(per):
            out[f"mae_ch{i}"] = float(v)
    return out


## 11. Output directory + structured logging

Every run gets its own folder with:

| File | Contents |
|---|---|
| `config.json` | Effective config (cfg/data_cfg/model_cfg/train_cfg, TAU, ALPHA) |
| `history.csv` | Per-epoch metrics |
| `steps.jsonl` | Per-step record (loss components, grad_norm, lr, skip counter) |
| `best.pt` | Lowest val_loss checkpoint |
| `last.pt` | Last-epoch checkpoint with full state for **PBS-walltime resume** |
| `recon/recon_e{NNN}.png` | Per-epoch input/mask/pred/error grids |

In [ ]:
save_dir = cfg.output_root / f"{cfg.tag}_{cfg.run_id}"
save_dir.mkdir(parents=True, exist_ok=True)
(save_dir / "recon").mkdir(exist_ok=True)
print(f"save_dir: {save_dir}")

with open(save_dir / "config.json", "w") as fp:
    json.dump({
        "cfg":       {k: str(v) for k, v in vars(cfg).items()},
        "data_cfg":  {k: str(v) for k, v in vars(data_cfg).items()},
        "model_cfg": vars(model_cfg),
        "train_cfg": vars(train_cfg),
        "TAU": TAU, "ALPHA": ALPHA,
    }, fp, indent=2)

# Header for history.csv. We let it grow; resume opens in append mode.
HISTORY_FIELDS = [
    "epoch", "step", "lr", "epoch_time_s",
    "train_loss", "train_recon", "train_sim", "train_std", "train_cov",
    "val_loss",   "val_recon",   "val_sim",   "val_std",   "val_cov",
    "grad_norm_mean_finite", "grad_norm_max", "n_inf_grad",
    "eff_rank", "r_max", "n_dead", "std_med", "D",
    "silhouette", "davies_bouldin",
    "mae_all", "mae_fg", "mae_bg", "mae_ch0", "mae_ch1", "mae_ch2",
]
hist_path = save_dir / "history.csv"
if not hist_path.exists():
    with open(hist_path, "w", newline="") as fp:
        csv.DictWriter(fp, fieldnames=HISTORY_FIELDS).writeheader()
steps_path = save_dir / "steps.jsonl"
last_ckpt  = save_dir / "last.pt"
best_ckpt  = save_dir / "best.pt"


## 12. Resume from `last.pt` (if present)

PBS walltime kills are common; this notebook resumes model + optim + scaler +
scheduler + epoch + global_step + RNG + best-so-far. Delete `last.pt` to start
fresh.

In [ ]:
start_epoch    = 1
global_step    = 0
best_val_loss  = float("inf")
best_epoch     = 0
n_inf_grad_run = 0

if last_ckpt.exists():
    ck = torch.load(last_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optim"])
    scheduler.load_state_dict(ck["sched"])
    if ck.get("scaler") is not None:
        scaler.load_state_dict(ck["scaler"])
    start_epoch    = int(ck["epoch"]) + 1
    global_step    = int(ck.get("global_step", 0))
    best_val_loss  = float(ck.get("best_val_loss", float("inf")))
    best_epoch     = int(ck.get("best_epoch", 0))
    n_inf_grad_run = int(ck.get("n_inf_grad_run", 0))
    rng = ck.get("rng", {})
    if "torch" in rng:  torch.set_rng_state(rng["torch"])
    if "cuda" in rng and rng["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(rng["cuda"])
    if "numpy" in rng: np.random.set_state(rng["numpy"])
    if "python" in rng: random.setstate(rng["python"])
    print(f"resumed from epoch {ck['epoch']}; starting at {start_epoch}; "
          f"best so far: {best_val_loss:.5f} @ epoch {best_epoch}")
else:
    print("no last.pt found — starting fresh")


## 13. Training loop

**Per step**: two forwards (one per view) sharing the same mask.

```
loss = w_recon * 0.5 * (L_recon(view1) + L_recon(view2))
     + w_vicreg * (λ_sim·MSE(p1,p2) + λ_std·hinge + λ_cov·cov)
```

Where `p1, p2 = projector(z_pool_v1), projector(z_pool_v2)`.

### Definitions

In [ ]:
def grid_image(t: torch.Tensor) -> np.ndarray:
    """Convert a (B,C,H,W) tensor in [0,1] to a 4x4 grid numpy array."""
    B = min(16, t.shape[0]); side = 4
    t = t[:B].clamp(0, 1).cpu().numpy()
    if t.shape[1] == 1: t = np.repeat(t, 3, axis=1)
    t = t.transpose(0, 2, 3, 1)
    H, W = t.shape[1], t.shape[2]
    grid = np.zeros((side*H, side*W, 3), dtype=np.float32)
    for i in range(B):
        r, c = divmod(i, side)
        grid[r*H:(r+1)*H, c*W:(c+1)*W] = t[i]
    return grid


def save_recon_png(model, loader, path: Path):
    model.eval()
    with torch.no_grad():
        x = next(iter(loader)).to(device)
        m = sample_mask(x)
        with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
            r, _ = model(x, m)
        r = r.float()
        err = (r - x).abs().clamp(0, 1)
        x_masked = x * (1.0 - m) + 0.5 * m
    fig, axes = plt.subplots(1, 4, figsize=(14, 4))
    for ax, img, t in zip(
        axes,
        [grid_image(x), grid_image(x_masked), grid_image(r), grid_image(err)],
        ["input", "masked", "pred", "|err|"],
    ):
        ax.imshow(img); ax.set_title(t); ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


def save_checkpoint(path: Path, *, epoch: int, **extra) -> None:
    state = {
        "model":      model.state_dict(),
        "optim":      optimizer.state_dict(),
        "sched":      scheduler.state_dict(),
        "scaler":     scaler.state_dict() if scaler is not None else None,
        "epoch":      epoch,
        "global_step": global_step,
        "best_val_loss": best_val_loss,
        "best_epoch":    best_epoch,
        "n_inf_grad_run": n_inf_grad_run,
        "model_cfg":  vars(model_cfg),
        "train_cfg":  vars(train_cfg),
        "TAU": TAU, "ALPHA": ALPHA,
        "rng": {
            "torch":  torch.get_rng_state(),
            "cuda":   torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            "numpy":  np.random.get_state(),
            "python": random.getstate(),
        },
    }
    state.update(extra)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(state, tmp)
    os.replace(tmp, path)


### Actuall training

In [ ]:


total_start = time.time()
for epoch in range(start_epoch, train_cfg.epochs + 1):
    model.train()
    t0 = time.time()
    sums = dict(loss=0.0, recon=0.0, sim=0.0, std=0.0, cov=0.0)
    grad_norms_finite = []
    grad_norm_max     = 0.0
    n_inf_grad_epoch  = 0
    n_steps           = 0

    pbar = tqdm(train_loader, desc=f"E{epoch} train", leave=False)
    for x in pbar:
        x = x.to(device, non_blocking=True)
        v1 = make_view(x, train_cfg.view_gain_lo, train_cfg.view_gain_hi, train_cfg.view_noise_std)
        v2 = make_view(x, train_cfg.view_gain_lo, train_cfg.view_gain_hi, train_cfg.view_noise_std)
        m  = sample_mask(x)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
            r1, z1 = model(v1, m)
            r2, z2 = model(v2, m)
            l_recon = 0.5 * (
                reconstruction_loss(r1, v1, m, train_cfg.loss_kind, TAU, ALPHA) +
                reconstruction_loss(r2, v2, m, train_cfg.loss_kind, TAU, ALPHA)
            )
        # VICReg in fp32 (project + loss).
        p1 = model.project(z1.float())
        p2 = model.project(z2.float())
        vic = vicreg_loss(p1, p2,
                          train_cfg.lambda_sim, train_cfg.lambda_std, train_cfg.lambda_cov)
        loss = train_cfg.w_recon * l_recon + train_cfg.w_vicreg * vic["total"]

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=train_cfg.grad_clip_norm)
        gn_f = float(gn)
        if math.isfinite(gn_f):
            grad_norms_finite.append(gn_f)
            grad_norm_max = max(grad_norm_max, gn_f)
        else:
            n_inf_grad_epoch += 1; n_inf_grad_run += 1
        scaler.step(optimizer); scaler.update()

        # Per-step record.
        rec = dict(
            step=global_step, epoch=epoch,
            loss=float(loss.item()),
            l_recon=float(l_recon.item()),
            l_sim=float(vic["sim"].item()),
            l_std=float(vic["std"].item()),
            l_cov=float(vic["cov"].item()),
            grad_norm=gn_f if math.isfinite(gn_f) else None,
            lr=float(optimizer.param_groups[0]["lr"]),
            n_inf_grad_run=n_inf_grad_run,
        )
        with open(steps_path, "a") as fp:
            fp.write(json.dumps(rec) + "\n")
        sums["loss"]  += rec["loss"]
        sums["recon"] += rec["l_recon"]
        sums["sim"]   += rec["l_sim"]
        sums["std"]   += rec["l_std"]
        sums["cov"]   += rec["l_cov"]
        n_steps       += 1
        global_step   += 1
        pbar.set_postfix(loss=f"{sums['loss']/n_steps:.4f}",
                         std=f"{sums['std']/n_steps:.4f}")

    train_t = time.time() - t0
    scheduler.step()

    # ----- Validation -----
    model.eval()
    vsums = dict(loss=0.0, recon=0.0, sim=0.0, std=0.0, cov=0.0); v_steps = 0
    with torch.no_grad():
        for x in val_loader:
            x = x.to(device, non_blocking=True)
            v1 = make_view(x, train_cfg.view_gain_lo, train_cfg.view_gain_hi, train_cfg.view_noise_std)
            v2 = make_view(x, train_cfg.view_gain_lo, train_cfg.view_gain_hi, train_cfg.view_noise_std)
            m  = sample_mask(x)
            with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
                r1, z1 = model(v1, m)
                r2, z2 = model(v2, m)
                l_rec = 0.5 * (
                    reconstruction_loss(r1, v1, m, train_cfg.loss_kind, TAU, ALPHA) +
                    reconstruction_loss(r2, v2, m, train_cfg.loss_kind, TAU, ALPHA)
                )
            p1 = model.project(z1.float()); p2 = model.project(z2.float())
            v = vicreg_loss(p1, p2,
                            train_cfg.lambda_sim, train_cfg.lambda_std, train_cfg.lambda_cov)
            l = train_cfg.w_recon * l_rec + train_cfg.w_vicreg * v["total"]
            vsums["loss"]  += float(l.item())
            vsums["recon"] += float(l_rec.item())
            vsums["sim"]   += float(v["sim"].item())
            vsums["std"]   += float(v["std"].item())
            vsums["cov"]   += float(v["cov"].item())
            v_steps += 1
    val_loss = vsums["loss"] / max(1, v_steps)

    # ----- Diagnostics (every diag_every epochs) -----
    do_diag = (epoch == 1) or (epoch % train_cfg.diag_every == 0) or (epoch == train_cfg.epochs)
    diag = dict(eff_rank=float("nan"), r_max=0, n_dead=-1, D=-1, std_med=float("nan"),
                silhouette=float("nan"), davies_bouldin=float("nan"))
    mae  = dict(mae_all=float("nan"), mae_fg=float("nan"), mae_bg=float("nan"),
                mae_ch0=float("nan"), mae_ch1=float("nan"), mae_ch2=float("nan"))
    if do_diag:
        feats = collect_pooled_features(model, val_loader, train_cfg.n_diag_batches)
        diag.update(collapse_metrics(feats))
        diag.update(cluster_metrics(feats, k=train_cfg.silhouette_k,
                                    n_max=train_cfg.silhouette_n_max, seed=cfg.seed))
        mae.update(recon_mae_split(model, val_loader, TAU))

    # ----- Print + persist -----
    cur_lr = optimizer.param_groups[0]["lr"]
    gn_mean = float(np.mean(grad_norms_finite)) if grad_norms_finite else float("nan")
    star = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss; best_epoch = epoch; star = " *"
        save_checkpoint(best_ckpt, epoch=epoch)
    save_checkpoint(last_ckpt, epoch=epoch)

    if (epoch == 1) or (epoch % train_cfg.recon_png_every == 0) or (epoch == train_cfg.epochs):
        save_recon_png(model, val_loader, save_dir / "recon" / f"recon_e{epoch:03d}.png")

    row = dict(
        epoch=epoch, step=global_step, lr=cur_lr, epoch_time_s=train_t,
        train_loss = sums["loss"]/max(1,n_steps),
        train_recon= sums["recon"]/max(1,n_steps),
        train_sim  = sums["sim"]/max(1,n_steps),
        train_std  = sums["std"]/max(1,n_steps),
        train_cov  = sums["cov"]/max(1,n_steps),
        val_loss   = val_loss,
        val_recon  = vsums["recon"]/max(1,v_steps),
        val_sim    = vsums["sim"]/max(1,v_steps),
        val_std    = vsums["std"]/max(1,v_steps),
        val_cov    = vsums["cov"]/max(1,v_steps),
        grad_norm_mean_finite=gn_mean, grad_norm_max=grad_norm_max,
        n_inf_grad=n_inf_grad_epoch,
        eff_rank=diag["eff_rank"], r_max=diag["r_max"], n_dead=diag["n_dead"],
        std_med=diag["std_med"], D=diag["D"],
        silhouette=diag["silhouette"], davies_bouldin=diag["davies_bouldin"],
        mae_all=mae["mae_all"], mae_fg=mae["mae_fg"], mae_bg=mae["mae_bg"],
        mae_ch0=mae["mae_ch0"], mae_ch1=mae["mae_ch1"], mae_ch2=mae["mae_ch2"],
    )
    with open(hist_path, "a", newline="") as fp:
        csv.DictWriter(fp, fieldnames=HISTORY_FIELDS).writerow(row)

    print(
        f"E{epoch:3d} | tr {row['train_loss']:.4f} (rec {row['train_recon']:.4f} "
        f"sim {row['train_sim']:.3f} std {row['train_std']:.3f} cov {row['train_cov']:.3f}) | "
        f"val {val_loss:.4f} | lr {cur_lr:.2e} | g {gn_mean:.2f} (inf {n_inf_grad_epoch}) | "
        f"eff_rank {diag['eff_rank']:.1f}/{diag['D']} | sil {diag['silhouette']:.3f} | "
        f"mae_fg {mae['mae_fg']:.4f} | t {train_t:.1f}s{star}"
    )

print(f"\ndone in {time.time()-total_start:.1f}s | "
      f"best val_loss={best_val_loss:.5f} at epoch {best_epoch}")


## 14. Post-training: reload best, save encoder weights

In [ ]:
ck = torch.load(best_ckpt, map_location=device, weights_only=False)
model.load_state_dict(ck["model"])
print(f"loaded best from epoch {ck['epoch']} (val_loss={ck.get('best_val_loss', float('nan')):.5f})")

# Encoder-only weights for downstream finetuning / clustering.
encoder_path = save_dir / "encoder.pt"
torch.save({
    "swinViT":   model.swinViT.state_dict(),
    "model_cfg": vars(model_cfg),
    "tag":       cfg.tag, "run_id": cfg.run_id,
    "TAU": TAU, "ALPHA": ALPHA,
}, encoder_path)
print(f"encoder weights → {encoder_path}")


## 15. Final report — collapse / clustering check

In [ ]:
feats_final = collect_pooled_features(model, val_loader, train_cfg.n_diag_batches)
final_collapse = collapse_metrics(feats_final)
final_cluster  = cluster_metrics(feats_final, k=train_cfg.silhouette_k,
                                 n_max=train_cfg.silhouette_n_max, seed=cfg.seed)
final_mae      = recon_mae_split(model, val_loader, TAU)

print("=== Final collapse / clustering metrics ===")
print(json.dumps({**final_collapse, **final_cluster, **final_mae}, indent=2, default=float))


In [ ]:
# === V4-FINAL-RECON-VIZ ===
# Reuse `viz_recon()` from the helpers cell on the SAME 4 overfit + 3
# unseen patches that the overfit-recon-viz cell visualised. Comparing
# the two figures (this one vs the OVERFIT one above) shows what the
# full training run learned on top of -- or destroyed compared to --
# the 4-patch overfit baseline.
#
# A genuinely useful pretraining run should reconstruct the UNSEEN
# patches markedly better here than after the 4-patch overfit; the
# OVERFIT patches may look slightly worse here (the full run does not
# memorise them) but the in-mask MAE should still be sensible.
viz_recon(
    model, x_fixed, mask_fixed, overfit_cfg.patch_indices,
    suptitle=f"FINAL recon -- {x_fixed.shape[0]} TRAIN patches "
             f"(same indices as overfit-recon-viz)",
    header="\nFINAL on overfit TRAIN patches (same indices as overfit-recon-viz):",
)

viz_recon(
    model, x_other, mask_other, viz_other_indices,
    suptitle="FINAL recon -- 3 UNSEEN random train patches "
             "(same indices as overfit-recon-viz)",
    header="\nFINAL on UNSEEN random patches (same indices as overfit-recon-viz):",
)

## V3 Sanity checks (post-training)

Reload the best checkpoint and run inference on the same fixed indices used
in the pre-training checks. Compares input vs reconstruction vs per-pixel
error, and plots the training/val loss curves from `history.csv`


In [ ]:
# === V3-SANITY-CHECKS POST ===
# Post-training sanity checks. Reloads the best checkpoint and visualises
# reconstruction quality on the same fixed indices used in the pre-train
# section, then on `unseen_idx` and `unseen_indices` (matches reference
# cells 38-40 of pretrain_simMIM_swin_v2_fixed.ipynb).
import matplotlib.pyplot as plt
import numpy as np
import torch

from types import SimpleNamespace
overfit_cfg    = SimpleNamespace(patch_indices=[3, 4, 50, 1000])
unseen_idx     = 100
unseen_indices = [200, 500, 1500, 2000]

# --- Reload best checkpoint
_best_path = save_dir / "best_model.pt"
if not _best_path.exists():
    # fall back to the explicit `best_ckpt` variable some training cells set
    try:
        _best_path = best_ckpt
    except NameError as e:
        raise FileNotFoundError(f"no best checkpoint found at {save_dir}") from e

ck = torch.load(_best_path, map_location=device, weights_only=False)
for _k in ("model",):
    if _k in ck:
        model.load_state_dict(ck[_k]); break
else:
    raise KeyError(f"no model state in checkpoint, keys={list(ck.keys())}")
_vl = ck.get("val_loss", ck.get("best_val_loss", float("nan")))
print(f"reloaded {_best_path.name}: epoch={ck.get('epoch','?')}  "
      f"val={float(_vl):.5f}")


def _resolve_fixed(ds, indices):
    base = getattr(ds, "_base", ds)
    return torch.stack([base[i] for i in indices]).to(device)


def _forward(x_target, seed_off):
    torch.manual_seed(cfg.seed + seed_off)
    model.eval()
    with torch.no_grad():
        mask_ = sample_mask(x_target)
        out = model(x_target, mask_)
        recon_ = out[0] if isinstance(out, tuple) else out
    return recon_.float(), mask_


def _grid(x_target, recon, mask, indices, suptitle):
    """4-row grid: input | mask | recon | per-pixel |err| with cyan mask
    contour."""
    err = (recon - x_target).abs()
    n = x_target.shape[0]
    rows = 4 if mask is not None else 3
    fig, axes = plt.subplots(rows, n, figsize=(3.2 * n, 3.0 * rows))
    if n == 1:
        axes = axes[:, None]
    for j in range(n):
        o = x_target[j].clamp(0, 1).cpu()
        r = recon[j].clamp(0, 1).cpu()
        e = err[j].mean(0).cpu()
        rgb_o = o.permute(1, 2, 0).numpy() if o.shape[0] >= 3 else o[0].numpy()
        rgb_r = r.permute(1, 2, 0).numpy() if r.shape[0] >= 3 else r[0].numpy()
        axes[0, j].imshow(rgb_o, cmap="gray")
        axes[0, j].set_title(f"input idx={indices[j]}"); axes[0, j].axis("off")
        row_off = 0
        if mask is not None:
            m = mask[j, 0].cpu()
            axes[1, j].imshow(m.numpy(), cmap="gray", vmin=0, vmax=1)
            axes[1, j].set_title(f"mask cov={mask[j].mean().item():.2f}")
            axes[1, j].axis("off")
            row_off = 1
        axes[1 + row_off, j].imshow(rgb_r, cmap="gray")
        axes[1 + row_off, j].set_title("recon"); axes[1 + row_off, j].axis("off")
        hm = axes[2 + row_off, j].imshow(e.numpy(), cmap="hot")
        if mask is not None:
            # cyan contour at the mask boundary (matches reference cell 39)
            axes[2 + row_off, j].contour(
                mask[j, 0].cpu().numpy(), levels=[0.5],
                colors=["cyan"], linewidths=1.2,
            )
            in_m  = mask[j, 0].cpu()
            in_mae = (e * in_m).sum() / max(in_m.sum().item(), 1.0)
            out_mae = (e * (1.0 - in_m)).sum() / max((1.0 - in_m).sum().item(), 1.0)
            axes[2 + row_off, j].set_title(
                f"|err| in={in_mae.item():.3f}  out={out_mae.item():.3f}"
            )
        else:
            axes[2 + row_off, j].set_title(f"|err| mean={e.mean().item():.3f}")
        axes[2 + row_off, j].axis("off")
    plt.suptitle(suptitle)
    plt.tight_layout(); plt.show()


# --- (1) overfit indices on the val split (reference cell 38)
x_target = _resolve_fixed(ds_val, overfit_cfg.patch_indices)
recon, mask = _forward(x_target, seed_off=1)
err = (recon - x_target).abs()
mae_per = err.mean(dim=(1, 2, 3))
for j, idx in enumerate(overfit_cfg.patch_indices):
    print(f"  overfit_idx={idx}: MAE={mae_per[j].item():.5f}")
_grid(x_target, recon, mask, overfit_cfg.patch_indices,
      "POST: overfit indices on val split")


# --- (2) single unseen index (reference cell 38)
x_target = _resolve_fixed(ds_val, [unseen_idx])
recon, mask = _forward(x_target, seed_off=2)
print(f"  unseen_idx={unseen_idx}: MAE={(recon - x_target).abs().mean().item():.5f}")
_grid(x_target, recon, mask, [unseen_idx], f"POST: unseen idx={unseen_idx}")


# --- (3) unseen grid (reference cell 40)
x_target = _resolve_fixed(ds_val, unseen_indices)
recon, mask = _forward(x_target, seed_off=3)
err = (recon - x_target).abs()
for j, idx in enumerate(unseen_indices):
    print(f"  unseen_idx={idx}: MAE={err[j].mean().item():.5f}")
_grid(x_target, recon, mask, unseen_indices, "POST: unseen grid")


# --- (4) Loss / eff_rank curves from history.csv
import csv as _csv
hist_path = save_dir / "history.csv"
if hist_path.exists():
    rows = list(_csv.DictReader(open(hist_path)))
    if rows:
        ep = [int(r["epoch"]) for r in rows]
        tr = [float(r["train_loss"]) for r in rows]
        vl = [float(r["val_loss"])   for r in rows]
        fig, ax = plt.subplots(1, 2, figsize=(13, 4))
        ax[0].plot(ep, tr, label="train"); ax[0].plot(ep, vl, label="val")
        ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
        ax[0].set_title("Loss curves")
        if "eff_rank" in rows[0] and rows[0]["eff_rank"] not in ("", "nan"):
            er = [float(r["eff_rank"]) if r["eff_rank"] not in ("", "nan") else float("nan")
                  for r in rows]
            ax[1].plot(ep, er); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("eff_rank")
            ax[1].set_title("Effective rank (representation diversity)")
        plt.tight_layout(); plt.show()
else:
    print(f"history.csv not found at {hist_path}")


# === V4-VICREG-EXTRAS ===
# VICReg-specific diagnostics: term magnitudes from history.csv (catches the
# `sim` term collapsing to 0) and projector embedding-norm histogram on the
# val split (catches mode collapse on z).
if hist_path.exists():
    rows = list(_csv.DictReader(open(hist_path)))
    if rows:
        # try several common column names so this works regardless of
        # whatever the training cell logs.
        candidates = (
            ("sim",       ("sim", "vic_sim", "lambda_sim", "loss_sim")),
            ("std",       ("std", "vic_std", "lambda_std", "loss_std")),
            ("cov",       ("cov", "vic_cov", "lambda_cov", "loss_cov")),
            ("recon",     ("rec", "recon", "loss_recon", "loss_rec")),
        )
        cols = []
        for label, names in candidates:
            for n in names:
                if n in rows[0] and rows[0][n] not in ("", "nan"):
                    cols.append((label, n)); break
        if cols:
            ep = [int(r["epoch"]) for r in rows]
            fig, ax = plt.subplots(figsize=(10, 4))
            for label, name in cols:
                ys = [float(r[name]) if r[name] not in ("", "nan") else float("nan")
                      for r in rows]
                ax.plot(ep, ys, label=f"{label} ({name})")
            ax.set_xlabel("epoch"); ax.set_ylabel("term magnitude")
            ax.legend(); ax.set_title("VICReg term magnitudes")
            plt.tight_layout(); plt.show()
        else:
            print("no VICReg term columns found in history.csv")

# Projector embedding-norm histogram across the val split.
model.eval()
zs = []
with torch.no_grad():
    for _bx in val_loader:
        _bx = _bx.to(device)
        _bm = sample_mask(_bx)
        _out = model(_bx, _bm)
        if not isinstance(_out, tuple) or len(_out) < 2:
            zs = None; break
        zs.append(_out[1].detach().cpu())
        if sum(z.shape[0] for z in zs) >= 1024:
            break
if zs:
    Z = torch.cat(zs, 0)
    norms = Z.norm(dim=1).numpy()
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(norms, bins=40)
    ax.set_xlabel("||z||"); ax.set_ylabel("count")
    ax.set_title(
        f"Projector embedding norms  median={np.median(norms):.3f}  "
        f"std={np.std(norms):.3f}  N={len(norms)}"
    )
    plt.tight_layout(); plt.show()
else:
    print("model does not return a projector output; skipping z-norm hist")
